# PyTorch/CUDA 12.9 setup
This notebook uses the refactored PyTorch DiscoRL implementation and no JAX/Haiku/Optax/RLax dependencies.

In [ ]:
import torch
import numpy as np
from ml_collections import config_dict
from disco_rl.agent import Agent
from disco_rl import types
from dm_env import specs

agent_settings = config_dict.ConfigDict(dict(
    update_rule_name="actor_critic",
    update_rule=dict(discount=0.99, value_cost=0.5),
    net_settings=dict(name="mlp", net_args=dict(hidden_sizes=(32, 32))),
    learning_rate=1e-3,
    max_abs_update=1.0,
    hyper_params=dict(entropy_cost=0.01, value_cost=0.5),
))
T, B = 6, 4
obs_spec = specs.Array((4,), np.float32)
act_spec = specs.BoundedArray((), np.int32, minimum=0, maximum=2)
agent = Agent(single_observation_spec=obs_spec, single_action_spec=act_spec, agent_settings=agent_settings, batch_axis_name=None)
state = agent.initial_learner_state()
rollout = types.ActorRollout(
    observations=torch.randn(T, B, 4),
    actions=torch.randint(0, 3, (T, B)),
    agent_outs={},
    rewards=torch.randn(T, B),
    discounts=torch.ones(T, B),
)
state, _, metrics = agent.learner_step(None, rollout, state, None, None, False)
print({k: float(v) for k, v in metrics.items()})
